# MAGI version 0.8 — flow continuum + fixed-line mixture energy head

v0.8 replaces v0.7.2's **categorical (per-bin) energy head** with a **mixture
energy head**: a conditional normalizing-flow *continuum* plus fixed-position
Gaussian *lines* whose widths are pinned to the X-IFU detector resolution
(4 eV FWHM). The motivation is spectroscopy-grade fidelity — v0.7.2's lines can
never be narrower than an energy bin (~20 keV at 511 keV over the CR log-E axis,
i.e. ~5000x the detector resolution), so it cannot represent a real line.

Model class: `CVAE_MixEnergy_ContPhi_TaskAdaptive` (`core/model.py`). Geometry is
unchanged from v0.7.2 (`[u_r_q, u_v_q, phi_r_q, phi_v_q]`, quantile-transformed).

**What is new relative to v0.7.2**

| piece | v0.7.2 | v0.8 |
|---|---|---|
| energy head | categorical over 512 log-bins | flow continuum + fixed-line Gaussian mixture |
| line width | one energy bin | pinned per line to 4 eV FWHM (`line_logsigma_trainable=False`) |
| continuum | — | conditional RQS flow, `core/flows.py` (24 bins, 3 transforms) |
| flow input warp | — | **CDF pre-warp** (Cycle 1): empirical-CDF -> N(0,1) so spline knots are density-proportional |
| gate supervision | — | auxiliary gate CE with **focal down-weighting** (Cycle 2, `gate_focal_gamma=2`) |
| latent prior | fixed `N(0, I)` | learnable conditional coupling prior `p(z\|cond)` (`core/priors.py`) + MC-KL |
| energy conditioning | — | `energy_flow_condition="z_cond"` — energy head sees `z`, keeping energy<->geometry coupled |

**Notebook layout** (same order as `MAGI_v0_7_2.ipynb`)

1. Dataset Import and Checks — load, physical features, diagnostics, **energy line detection** (v0.8-specific).
2. Standardization and Preprocessing — feature dataframe, gate targets, split/conditioning, **CDF warp knots**.
3. Training with MAGI Package — model config, build, callbacks, fit, history.
4. Generation and Validation — spectra, line-integral recovery, per-variable residuals, pairgrid, covariance/correlation.
5. Saving Model.
6. Generating Input Files for Geant4.
7. Appendix — flow vs Gaussian continuum (synthetic).

**One source per run.** Unlike the previous version of this notebook (which looped
over sources), each section works on a single `SOURCE` so it reads like v0.7.2 and
the `magi.report_*` helpers apply directly. Set `SOURCE = "CR"` or `"Small"` at the
top of the data cell and re-run. The unattended both-sources equivalent is
`tools/run_v0_8_real.py` (training + spectra) and `tools/plot_v0_8_real_corr.py`
(multivariate validation).

**Status.** See `docs/v0.8_v072_comparison.md` for the full v0.7.2<->v0.8 comparison
and the Cycle 0 -> 1 -> 2 progression; `docs/v0.8_fixing_plan.md` for the fix plan.
Short version: the coupling prior and the continuum are validated on real data, the
lines are good on Small and still heterogeneous on CR, and v0.7.2 remains the beta
fallback until CR line recovery closes.

In [ ]:
# ==========================================================
# Imports
# Force CPU BEFORE importing magi: tensorflow-metal is ~8-10% slower than CPU
# for this tfp op mix on the M1, and magi pulls in tensorflow_probability,
# which would otherwise initialize the Metal device - after which hiding the
# GPU no longer takes effect. (Same choice as MAGI_v0_7_2.ipynb.)
# ==========================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # reduce TensorFlow log verbosity

import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import wasserstein_distance

# ==========================================================
# TensorFlow / Keras
# ==========================================================
import tensorflow as tf
tf.config.set_visible_devices([], "GPU")

from tensorflow import keras

print("TensorFlow version:", tf.__version__)
print("Keras version     :", keras.__version__)
print("\nPhysical devices  :", tf.config.list_physical_devices())
print("Visible GPU(s)    :", tf.config.get_visible_devices("GPU"))

In [ ]:
import magi

print(magi.__file__)
print(magi.__version__)
print(magi.CVAE_CatEnergy_ContPhi_TaskAdaptive)   # v0.7.2 head, for reference
print(magi.CVAE_MixEnergy_ContPhi_TaskAdaptive)   # v0.8 head, used here

magi.initialize_environment(seed=42, cpu_only=True)
magi.set_plot_theme("light")

## Dataset Import and Checks

One file per background component (see `CLAUDE.md` for the data layout). This
notebook runs **one source at a time** — pick it below and re-run the notebook
for the other one. CR and Small (K-40) are the two sources in scope for v0.8;
the Radio/Torio (Ra-226 / Th-232) files are deferred to a later phase.

In [ ]:
TRAINING_DATA_DIR = "/Volumes/X10Pro/MAGI/TrainingData"

SOURCE_FILES = {
    "CR":    f"{TRAINING_DATA_DIR}/alloutputDSCryoSphereCR.dat",
    "Small": f"{TRAINING_DATA_DIR}/alloutputDSCryoSphereSmall.dat",
}

# ---- pick the source for this run -------------------------------------
SOURCE = "CR"          # "CR" or "Small"
# -----------------------------------------------------------------------

center = (0.0, 0.0, -507.66)
R = 100.0

X_IFU_RESOLUTION_EV = 4.0   # X-IFU energy resolution (FWHM), pins the line widths

df = magi.load_detector_table(
    filepath=SOURCE_FILES[SOURCE],
    sep=r"\s+",
)

magi.report_basic_table_checks(df)

prep = magi.build_physical_features(
    df,
    center=center,
    radius=R,
)

magi.print_physical_summary(prep)

# Base diagnostic objects
feat_base = prep["features"]
raw = prep["raw"]
proj = prep["projected"]
normdir = prep["normalized_direction"]
derived = prep["derived"]

E_all = feat_base["Energy"].to_numpy()
print(f"\n{SOURCE}: {len(df):,} rows, {E_all.size:,} valid energies, "
      f"E in [{E_all.min():.6f}, {E_all.max():.4f}] MeV "
      f"({np.log10(E_all.max() / E_all.min()):.1f} decades)")

In [ ]:
# ==========================================================
# Energy diagnostics from base physical features
# The v0.8 energy head models y = log10(E) directly (no binning), so the
# log-E shape below IS the target density the flow has to reproduce.
# ==========================================================

magi.plot_dist(feat_base["logE"], "log10(E)")
magi.plot_dist(feat_base["logE"], "log10(E) [log density]", yscale="log")
magi.plot_dist_by_class(feat_base, "logE", selected_class="gamma")
magi.plot_dist_by_class(feat_base, "logE", selected_class="e-")

# ==========================================================
# Raw / projected geometry
# ==========================================================

magi.plot_dist(raw["x"], "x raw")
magi.plot_dist(raw["y"], "y raw")
magi.plot_dist(raw["z"], "z raw")
magi.plot_dist(raw["r"], "r raw")

magi.plot_dist(proj["x"], "x projected")
magi.plot_dist(proj["y"], "y projected")
magi.plot_dist(proj["z"], "z projected")

magi.plot_dist(normdir["vx"], "vx")
magi.plot_dist(normdir["vy"], "vy")
magi.plot_dist(normdir["vz"], "vz")

# ==========================================================
# Raw physical geometry variables (unchanged from v0.7.2)
# ==========================================================

magi.plot_dist(feat_base["u_r"], "u_r raw = cos(theta_r)", range_=(-1, 1))
magi.plot_dist(feat_base["u_v"], "u_v raw = cos(theta_v)", range_=(-1, 1))
magi.plot_dist(feat_base["phi_r"], "phi_r raw", range_=(-np.pi, np.pi))
magi.plot_dist(feat_base["phi_v"], "phi_v raw", range_=(-np.pi, np.pi))

plt.show()

### Energy line detection (v0.8-specific)

The mixture head needs to be told *where* the lines are: their positions are
fixed inputs (`line_positions_y`), not learned. So before any preprocessing we
detect statistically significant peaks in the real spectrum and match them
against the mass model's candidate line table
(`CandidateLines/CANDIDATE_ENERGY_LINES_SRON_CCNwithXFDM_NoShield_FlowerCryoAC_fixed.json`,
built by `tools/build_candidate_lines_from_geant4.py` from Geant4's own
fluorescence database (G4EMLOW7.3/fluor_Bearden, for every element actually
present in the GDML mass model) and decay-level database (PhotonEvaporation5.2,
for the K-40 / Ra-226 / Th-232 / Rn-222 sources) — not values hand-typed from
outside literature).

Expected: the instrumental fluorescence lines show up as `matched_lines`; the
source decay lines mostly show up as `unmatched_candidates`, since the sources
sit in bulk shielding and their primary gammas are Compton-degraded before they
cross the CryoSphere.

In [ ]:
CANDIDATE_LINES_FILE = (
    "/Volumes/X10Pro/MAGI/CandidateLines/"
    "CANDIDATE_ENERGY_LINES_SRON_CCNwithXFDM_NoShield_FlowerCryoAC_fixed.json"
)

candidate_payload = magi.load_candidate_energy_lines(CANDIDATE_LINES_FILE)
candidate_lines = candidate_payload["lines"]

print(f"Loaded {candidate_payload['n_lines']} candidate lines "
      f"for mass model: {candidate_payload['mass_model']}")
print("Sources covered:", candidate_payload["sources"])

In [ ]:
line_result = magi.detect_energy_lines(
    E_all,
    binning_mode="log_fixed_count",
    n_bins=1024,
    prominence_factor=3.0,
    window=5,
    candidate_lines=candidate_lines,
)

magi.print_detected_energy_lines(line_result)

# Weak matches (single- to double-digit real counts out of millions of events)
# are filtered out before they feed the mixture head: a line slot the gate can
# never populate reliably only adds a spurious mixture component. Phase A above
# still reports every statistical match, for transparency; this filter is
# layered on top, here, not inside detect_energy_lines.
matched_all = line_result["matched_lines"]
matched = [m for m in matched_all if m["count"] >= 100]
dropped = [m for m in matched_all if m["count"] < 100]

print(f"\n{len(matched_all)} matched lines; {len(matched)} feed the mixture head "
      f"(count>=100): {[m['label'] for m in matched]}")
if dropped:
    print(f"  dropped as too weak (count<100): "
          f"{[(m['label'], int(m['count'])) for m in dropped]}")

In [ ]:
def plot_line_detection(name, E, result):
    edges = np.asarray(result["energy_bins"])
    centres = 0.5 * (edges[:-1] + edges[1:])
    counts, _ = np.histogram(E, bins=edges)

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.step(centres, counts, where="mid", color="#2b6cb0", label="real spectrum")

    for m in result["matched_lines"]:
        ax.axvline(m["candidate_energy_mev"], color="#38a169", lw=1, ls="--")
    for c in result["unmatched_candidates"]:
        ax.axvline(c["candidate_energy_mev"], color="#a0aec0", lw=0.7, ls=":")

    peak_e = [p["energy_mev"] for p in result["detected_peaks"]]
    peak_c = [p["count"] for p in result["detected_peaks"]]
    ax.scatter(peak_e, peak_c, color="#e53e3e", zorder=5, label="detected peaks")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Energy [MeV]")
    ax.set_ylabel("counts / bin")
    ax.set_title(
        f"{name}: {result['n_matched']} matched / {result['n_candidate_lines']} "
        "candidate lines (green = matched, grey = expected-but-undetected)"
    )
    ax.legend(fontsize=8)
    plt.show()


plot_line_detection(SOURCE, E_all, line_result)

## Standardization and Preprocessing

Same pipeline as v0.7.2 — `build_feature_dataframe` -> `filter_particle_types_
continuous_geometry` -> `split_feature_data` -> `scale_continuous_features` ->
`build_conditioning_and_weights` -> `build_tf_datasets` — with three v0.8
additions:

1. `energy_transform="log10"`: energy enters the model as the **continuous**
   column `energy_y = log10(E)` instead of a bin index. The `energy_bins` are
   still built (needed for gate targets, line detection and the validation
   histograms), but the model never sees them.
2. **Gate targets**: `build_gate_targets` labels each event as continuum or as
   belonging to one of the matched lines. They ride along as extra `y_cont`
   columns and supervise the mixture gate (auxiliary CE).
3. **CDF warp knots** (Cycle 1): a monotone empirical-CDF -> N(0,1) map fitted on
   `energy_y`, used as the flow's input standardization.

In [ ]:
feature_pack = magi.build_feature_dataframe(
    prep,
    energy_binning_mode="log_fixed_count",
    n_bins=512,
    geometry_transform="quantile_u_r_u_v_phi_r_phi_v",
    n_quantiles=10000,
    random_state=42,
    # v0.8: continuous energy target y = log10(E) alongside the bin index
    energy_transform="log10",
)

magi.report_feature_dataframe(feature_pack)

energy_bins = feature_pack["energy_bins"]
n_energy_bins = feature_pack["n_energy_bins"]

quantile_transformers = feature_pack["quantile_transformers"]
qt_u_r = quantile_transformers["qt_u_r"]
qt_u_v = quantile_transformers["qt_u_v"]
qt_phi_r = quantile_transformers["qt_phi_r"]
qt_phi_v = quantile_transformers["qt_phi_v"]

geometry_metadata = feature_pack["geometry_metadata"]

print("Feature columns:", feature_pack["feat"].columns.tolist())
print("Geometry metadata:", geometry_metadata)

In [ ]:
# ==========================================================
# v0.8 line inputs + gate targets
#   line_positions_y : fixed line centres in y = log10(E) space
#   gate_targets     : per-event one-hot over [continuum, line_0, ..., line_{L-1}]
# ==========================================================

line_positions_mev = np.array([m["candidate_energy_mev"] for m in matched], dtype=np.float64)
line_positions_y = np.log10(line_positions_mev).astype(np.float32)

E_full = feature_pack["filtered_prep"]["features"]["Energy"].to_numpy()

gate_targets = magi.build_gate_targets(
    E_full,
    feature_pack["energy_bins"],
    matched,
    resolution_mev=None,
)

feat = feature_pack["feat"].copy()
for j in range(gate_targets.shape[1]):
    feat[f"gate_target_{j}"] = gate_targets[:, j]

cont_cols = ("u_r_q", "u_v_q", "phi_r_q", "phi_v_q", "energy_y") + tuple(
    f"gate_target_{j}" for j in range(gate_targets.shape[1])
)

gate_mass = gate_targets.mean(axis=0)
print("line_positions_y (log10 E):", line_positions_y)
print("line labels             :", [m["label"] for m in matched])
print("y_cont columns          :", cont_cols)
print("gate slot masses        : continuum=%.5f | %s" % (
    gate_mass[0],
    " ".join(f"{m['label']}={f:.5f}" for m, f in zip(matched, gate_mass[1:]))))
print("\nThis imbalance (~99% continuum) is exactly what the focal term in the "
      "gate loss is for - see the model cell below.")

In [ ]:
dataset_pack = magi.filter_particle_types_continuous_geometry(
    feat=feat,
    prob_threshold=1e-5,
    cont_cols=cont_cols,
)

magi.report_continuous_geometry_features(dataset_pack)

split_pack = magi.split_feature_data(
    dataset_pack,
    test_size_total=0.30,
    val_size_from_temp=0.50,
    random_state=42,
)

magi.report_split_summary(split_pack, n_types=dataset_pack["n_types"])

scaled_pack = magi.scale_continuous_features(
    split_pack,
    scale_cols=(),
)

magi.report_scaled_features(scaled_pack)

condition_pack = magi.build_conditioning_and_weights(
    scaled_pack,
    n_types=dataset_pack["n_types"],
    idx_to_type=dataset_pack["idx_to_type"],
    alpha=0.5,
)

magi.report_conditioning(condition_pack, n_types=dataset_pack["n_types"])

tf_pack = magi.build_tf_datasets(
    condition_pack,
    batch_size=4096,
    shuffle_buffer_cap=200_000,
)

magi.report_tf_datasets(tf_pack)

magi.report_energy_binning_diagnostics(
    energy_bins=feature_pack["energy_bins"],
    E_idx=dataset_pack["E_idx"],
    E_values=feature_pack["filtered_prep"]["features"]["Energy"].to_numpy(),
    idx_train=split_pack["idx_train"],
    energy_binning_mode=feature_pack["energy_config"]["mode"],
    bin_width=feature_pack["energy_config"]["bin_width"],
    n_bins=feature_pack["energy_config"]["n_bins"],
    min_counts=feature_pack["energy_config"]["min_counts"],
)

In [ ]:
# ==========================================================
# Recover all main objects from package outputs - v0.8
# ==========================================================

feat = dataset_pack["feat"]

# model-side continuous block: [u_r_q, u_v_q, phi_r_q, phi_v_q, energy_y, gate_*]
X_cont_train_s = tf_pack["X_cont_train_s"]
X_cont_val_s = tf_pack["X_cont_val_s"]
X_cont_test_s = tf_pack["X_cont_test_s"]

X_cont_train = tf_pack["X_cont_train"]
X_cont_val = tf_pack["X_cont_val"]
X_cont_test = tf_pack["X_cont_test"]

# energy bin indices: kept for diagnostics/validation only - the v0.8 head does
# NOT consume them (it reads energy_y, column 4 of X_cont).
E_train = tf_pack["E_train"]
E_val = tf_pack["E_val"]
E_test = tf_pack["E_test"]

cond_train = tf_pack["cond_train"]
cond_val = tf_pack["cond_val"]
cond_test = tf_pack["cond_test"]

train_ds = tf_pack["train_ds"]
val_ds = tf_pack["val_ds"]
test_ds = tf_pack["test_ds"]

type_weights = tf_pack["type_weights"]

n_types = dataset_pack["n_types"]
idx_to_type = dataset_pack["idx_to_type"]
type_probs = dataset_pack["type_probs"]

cont_cols = dataset_pack["cont_cols"]
Y_CONT_DIM = X_cont_train_s.shape[1]
N_LINES = int(len(matched))

# raw physical references, per split
features_filtered = feature_pack["filtered_prep"]["features"]

E_all_filtered = features_filtered["Energy"].to_numpy()
u_r_all_raw = features_filtered["u_r"].to_numpy()
u_v_all_raw = features_filtered["u_v"].to_numpy()
phi_r_all_raw = features_filtered["phi_r"].to_numpy()
phi_v_all_raw = features_filtered["phi_v"].to_numpy()

idx_train = split_pack["idx_train"]
idx_val = split_pack["idx_val"]
idx_test = split_pack["idx_test"]

E_train_raw = E_all_filtered[idx_train]
E_val_raw = E_all_filtered[idx_val]
E_test_raw = E_all_filtered[idx_test]

print("Y_CONT_DIM:", Y_CONT_DIM, "(4 geometry + 1 energy_y + %d gate slots)" % (N_LINES + 1))
print("cont_cols :", cont_cols)
print("n_types   :", n_types, idx_to_type)
print("n_lines   :", N_LINES)
print("Available quantile transformers:", list(quantile_transformers.keys()))

In [ ]:
# ==========================================================
# Split sanity: the three splits must be statistically identical
# ==========================================================

df_train_plot = dataset_pack["feat"].iloc[split_pack["idx_train"]].copy()
df_val_plot   = dataset_pack["feat"].iloc[split_pack["idx_val"]].copy()
df_test_plot  = dataset_pack["feat"].iloc[split_pack["idx_test"]].copy()

df_train_plot["split"] = "train"
df_val_plot["split"]   = "val"
df_test_plot["split"]  = "test"

df_all_splits = pd.concat(
    [df_train_plot, df_val_plot, df_test_plot],
    axis=0,
    ignore_index=True,
)

model_space_cols = ["energy_y", "u_r_q", "u_v_q", "phi_r_q", "phi_v_q"]

magi.plot_pairwise_sample(
    df_all_splits,
    cols=model_space_cols,
    class_col="split",
    sample_size=8000,
)

magi.plot_correlation_matrix(
    df_all_splits,
    model_space_cols,
    method="pearson",
)

plt.show()

In [ ]:
# ==========================================================
# Real-data structure the model has to reproduce: correlation, covariance and
# the full pairgrid over the physical variables. These are the reference for
# the identical plots on generated events in the validation section.
# ==========================================================

corr_df = feature_pack["filtered_prep"]["features"].copy()
corr_df["logE"] = np.log10(corr_df["Energy"].to_numpy())

corr_cols_phys = ["logE", "u_r", "u_v", "phi_r", "phi_v"]

corr_real = magi.plot_correlation_matrix(corr_df, corr_cols_phys, method="pearson")
cov_real = magi.plot_covariance_matrix(corr_df, corr_cols_phys)

magi.plot_pairgrid_physics(
    corr_df,
    cols=corr_cols_phys,
    class_col="ParticleName",
    sample_size=8000,
    lower_mode="scatter",
    bins=30,
    contour_levels=8,
    figsize_scale=2.4,
    theme="light",
    palette="tab10",
)

plt.show()

In [ ]:
# ==========================================================
# CDF pre-warp knots for the flow continuum  (v0.8 Cycle 1)
#
# The RQS flow warps a FIXED interval [-B, B] (B = interval_half_width = 5) in
# standardized space, with frozen N(0,1) tails outside it. With plain affine
# standardization (y - mean)/std that interval is allocated by *variance*, not
# by where the events are:
#   - CR: the muon tail (to ~1e6 MeV) inflates the std, so the low-energy bulk
#     occupies a tiny slice of [-5, 5] and too few spline knots land on the
#     Compton edge -> it comes out over-smoothed.
#   - Small: the bulk is narrow, so the sparse low-energy tail lands near/beyond
#     B=5 -> the frozen normal tails take over and the tail density collapses.
#
# fit_cdf_warp_knots builds a monotone empirical-CDF -> N(0,1) map instead, so the
# knots are density-proportional: every quantile of the data gets comparable
# spline resolution and nothing falls outside the modelled interval. Validated on
# the --sparse-tail synthetic: deep-tail window ratio 0.00 (affine) -> 0.96 (cdf).
# See docs/v0.8_v072_comparison.md sec 6(b) and sec 9.
# ==========================================================

energy_y_all = feat["energy_y"].to_numpy()

warp_y_knots, warp_z_knots = magi.fit_cdf_warp_knots(
    energy_y_all,
    n_knots=256,
    eps=1e-4,
)

y_mean_affine = float(energy_y_all.mean())
y_scale_affine = float(energy_y_all.std())

print(f"CDF warp: {warp_y_knots.size} knots over "
      f"y in [{warp_y_knots[0]:.3f}, {warp_y_knots[-1]:.3f}] "
      f"-> z in [{warp_z_knots[0]:.2f}, {warp_z_knots[-1]:.2f}]")
print(f"affine reference (NOT used): y_mean={y_mean_affine:.3f} y_scale={y_scale_affine:.3f}")

# What the two standardizations do to the same data: the fraction of [-5, 5]
# that the physically interesting bulk actually occupies.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

axes[0].plot(warp_y_knots, warp_z_knots, color="#2b6cb0", lw=2, label="CDF warp")
axes[0].plot(warp_y_knots, (warp_y_knots - y_mean_affine) / y_scale_affine,
             color="#dd6b20", lw=2, ls="--", label="affine (y-mean)/std")
axes[0].axhline(5, color="0.5", lw=0.8, ls=":")
axes[0].axhline(-5, color="0.5", lw=0.8, ls=":")
axes[0].set_xlabel("y = log10(E)")
axes[0].set_ylabel("standardized flow input")
axes[0].set_title(f"{SOURCE}: warp map (dotted = flow interval +/- B)")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

for lbl, z, c in [("CDF warp", np.interp(energy_y_all, warp_y_knots, warp_z_knots), "#2b6cb0"),
                  ("affine", (energy_y_all - y_mean_affine) / y_scale_affine, "#dd6b20")]:
    axes[1].hist(np.clip(z, -8, 8), bins=200, histtype="step", lw=1.6, color=c,
                 density=True, label=lbl)
axes[1].axvline(5, color="0.5", lw=0.8, ls=":")
axes[1].axvline(-5, color="0.5", lw=0.8, ls=":")
axes[1].set_xlabel("standardized flow input")
axes[1].set_ylabel("density")
axes[1].set_title("where the events land inside the spline interval")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Training with MAGI Package

### Model v0.8 `CVAE_MixEnergy_ContPhi_TaskAdaptive`

Energy head = **conditional RQS-flow continuum** + **fixed-position Gaussian
lines**, mixed by a learned gate:

$$p(y \mid z, c) \;=\; \pi_0(z,c)\, p_{\mathrm{flow}}(y \mid z, c) \;+\;
\sum_{\ell=1}^{L} \pi_\ell(z,c)\, \mathcal{N}\!\left(y; y_\ell, \sigma_\ell\right)$$

with the line centres $y_\ell$ fixed from Phase A and the widths $\sigma_\ell$
**pinned** to 4 eV FWHM (converted into log10-E space per line by
`line_logsigma_from_resolution`, then frozen).

Key configuration choices, and why:

- `continuum_mode="flow"`, `continuum_flow_bins=24`, `continuum_flow_transforms=3` —
  a K=1 parametric Gaussian continuum cannot be simultaneously sharp at the peak
  and near-zero in the empty regions (see the synthetic appendix).
- `continuum_flow_warp="cdf"` with the knots fitted above — **Cycle 1**.
- `energy_flow_condition="z_cond"` + `prior="coupling"` — the energy head reads the
  latent `z`, which is what keeps energy correlated with geometry; a fixed `N(0,I)`
  prior would then leak per-event energy through `z` and degrade generation (the
  aggregated-posterior/prior mismatch), so the prior is a learnable conditional
  coupling flow `p(z|cond)` trained with an MC-KL. See
  `docs/v0.8_learnable_prior_plan.md`.
- `w_gate_aux=2.0`, `gate_focal_gamma=2.0` — **Cycle 2**. The gate CE is dominated
  by the ~99% continuum majority (printed above), so rare fluorescence lines never
  get routed. The focal factor $(1-p)^\gamma$ down-weights the easy majority and is
  self-limiting, unlike inverse-frequency class weights, which blew up (~12x at
  0.2%/line -> 44x over-routing). `gate_class_weights` (per-line manual weights)
  exists as the next lever and is checkpoint-safe, but is left `None` here.
- `line_logsigma_trainable=False` — the widths are a **detector property**, not
  something to fit; leaving them free lets the model widen lines to absorb
  continuum.

The v0.7.2 `TaskAdaptiveLossScheduler` is intentionally **not** used: it monitors a
categorical-energy quality metric that does not exist for the mixture head. Early
stopping + LR annealing on `val_loss` are kept.

In [ ]:
# ==========================================================
# Configuration for later saving and reference - v0.8
# (model.to_generation_config() is the authoritative version written into the
#  checkpoint; this dict documents the run and its preprocessing.)
# ==========================================================

EPOCHS = 40
LEARNING_RATE = 2e-4

line_logsigma_init = magi.line_logsigma_from_resolution(
    line_positions_mev,
    X_IFU_RESOLUTION_EV,
    fwhm=True,
)

model_config = {
    "model_class": "CVAE_MixEnergy_ContPhi_TaskAdaptive",
    "n_types": n_types,
    "line_positions_y": line_positions_y.tolist(),

    "latent_dim": 8,
    "hidden": [128, 128, 64],
    "beta": 0.2,

    # continuum: flow + CDF pre-warp (Cycle 1)
    "continuum_mode": "flow",
    "continuum_flow_bins": 24,
    "continuum_flow_transforms": 3,
    "continuum_flow_warp": "cdf",

    # coupling-preserving conditioning + learnable prior
    "energy_flow_condition": "z_cond",
    "prior": "coupling",

    # lines + gate (Cycle 2)
    "w_gate_aux": 2.0,
    "gate_focal_gamma": 2.0,
    "gate_class_weights": None,
    "line_logsigma_trainable": False,
    "x_ifu_resolution_ev": X_IFU_RESOLUTION_EV,
}

preprocessing_metadata = {
    "source": SOURCE,
    "geometry_transform": "quantile_u_r_u_v_phi_r_phi_v",
    "energy_transform": "log10",
    "cont_cols": list(cont_cols),
    "y_cont_dim": Y_CONT_DIM,

    "energy_bins": energy_bins,
    "type_probs": type_probs,
    "idx_to_type": idx_to_type,
    "n_types": n_types,

    "energy_binning_mode": feature_pack["energy_config"]["mode"],
    "energy_config": feature_pack["energy_config"],

    "matched_lines": [
        {"label": m["label"], "candidate_energy_mev": float(m["candidate_energy_mev"]),
         "count": int(m["count"])}
        for m in matched
    ],
    "line_positions_y": line_positions_y.tolist(),
    "line_logsigma_init": np.asarray(line_logsigma_init).tolist(),

    "geometry_metadata": geometry_metadata,
    "quantile_transformer_keys": list(quantile_transformers.keys()),
}

print("pinned per-line logsigma (log10-E space):", np.round(np.asarray(line_logsigma_init), 4))

In [ ]:
magi.initialize_environment(seed=42, cpu_only=True, quiet=True)

model = magi.CVAE_MixEnergy_ContPhi_TaskAdaptive(
    n_types=n_types,
    line_positions_y=line_positions_y,

    latent_dim=model_config["latent_dim"],
    hidden=tuple(model_config["hidden"]),
    beta=model_config["beta"],

    # --- flow continuum with CDF pre-warp (Cycle 1) ---
    continuum_mode="flow",
    continuum_flow_bins=model_config["continuum_flow_bins"],
    continuum_flow_transforms=model_config["continuum_flow_transforms"],
    continuum_flow_warp="cdf",
    continuum_flow_warp_y_knots=warp_y_knots,
    continuum_flow_warp_z_knots=warp_z_knots,

    # --- z-conditioning + learnable coupling prior ---
    energy_flow_condition="z_cond",
    prior="coupling",

    # --- lines + focal gate supervision (Cycle 2) ---
    w_gate_aux=model_config["w_gate_aux"],
    gate_focal_gamma=model_config["gate_focal_gamma"],
    gate_class_weights=model_config["gate_class_weights"],
    line_logsigma_init=line_logsigma_init,
    line_logsigma_trainable=False,
)

magi.compile_model(model, learning_rate=LEARNING_RATE)

# v0.7.2 callback stack: early stopping + LR anneal on val_loss.
callbacks = magi.build_default_callbacks(
    monitor="val_loss",
    early_patience=8,
    lr_patience=6,
    factor=0.5,
    min_lr=1e-5,
    verbose=1,
)

# DUMMY BUILD FOR STRUCTURE INSPECTION
dummy_z = tf.zeros((1, model.latent_dim), dtype=tf.float32)
dummy_cond = tf.zeros((1, n_types), dtype=tf.float32)
_ = model.decode(dummy_z, dummy_cond)

magi.print_model_tree_with_params(model)
magi.print_duplicate_trainable_variables(model)

In [ ]:
magi.print_model_structure(model)

In [ ]:
# ~48 min for CR / ~33 min for Small at 40 epochs on the M1 CPU.
# verbose=2 keeps one line per epoch (readable in a long run).
t0 = time.time()

history = magi.fit_model(
    model=model,
    train_ds=train_ds,
    val_ds=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=2,
)

n_epochs_run = len(history.history["loss"])
print(f"\n{SOURCE}: trained {n_epochs_run} epochs in {time.time() - t0:.0f}s")
print(f"  val energy_mixture_nll = {history.history['val_energy_mixture_nll'][-1]:.4f}")
print(f"  val gate_aux_loss      = {history.history['val_gate_aux_loss'][-1]:.4f}")
print(f"  val kl                 = {history.history['val_kl'][-1]:.4f}")
print(f"  pinned per-line logsigma = {np.round(model._line_logsigma_clipped().numpy(), 3)}")

In [ ]:
# v0.8 metric set: energy_mixture_nll replaces v0.7.2's energy_ce, and
# gate_aux_loss is the line-routing supervision to watch.
magi.plot_history(
    history,
    keys=[
        "loss",
        "rec",
        "kl",
        "nll",
        "energy_mixture_nll",
        "gate_aux_loss",
        "ur_nll",
        "uv_nll",
        "phi_r_nll",
        "phi_v_nll",
        "sigma_reg",
    ],
)

## Generation and Validation

Three levels, in increasing strictness:

1. **Energy marginal** — the spectrum overlay plus the per-line integral recovery
   (`compute_line_integral_recovery`): the ratio of generated to real events in a
   window around each known line. This is where v0.8 is still being tuned.
2. **Per-variable marginals** — `compare_hist_with_residuals` over every physical
   and model-space variable, plus Wasserstein distances and the geometric
   constraint checks (|r| = R, ||v|| = 1).
3. **Joint structure** — pairgrid and covariance/correlation triptychs
   (real | generated | gen-real). This is the test the learnable coupling prior
   exists for: energy<->geometry correlations must survive generation.

In [ ]:
### SKIP MODEL TRAINING AND DIRECTLY LOAD A TRAINED MODEL FOR GENERATION AND EVALUATION
#
# The v0.8 checkpoint is self-describing: model.to_generation_config() stores the
# line positions, continuum_mode, flow + warp knots, prior type, focal gamma and
# per-line widths, so the loader rebuilds exactly this architecture. Just read the
# saved JSON back - no need to retype the config as in v0.7.2.

save_dir = f"trained_models/v0_8_{SOURCE}"
model_name = f"mix_{SOURCE}"

with open(os.path.join(save_dir, f"{model_name}_config.json")) as f:
    saved_config = json.load(f)

model = magi.load_task_adaptive_model_for_generation(
    save_dir=save_dir,
    model_name=model_name,
    model_config=saved_config,
    energy_bins=energy_bins,
    u_v_bins=None,              # v0.8: not used
    n_types=n_types,
    type_weights=type_weights,
    radius=R,
    compile_model_fn=magi.compile_model,
    verbose=1,
)

In [ ]:
# ==========================================================
# Generate and reconstruct physics
# Generation is chunked so a large NGEN stays memory-safe.
# ==========================================================

NGEN = int(len(X_cont_test))
# NGEN = int(1e6)
CHUNK = 500_000

def generate_physics(model, n_gen, chunk=CHUNK):
    parts = []
    done = 0
    while done < n_gen:
        m = min(chunk, n_gen - done)
        gen_raw = magi.generate_latent_outputs(
            model=model,
            n_samples=m,
            type_probs=type_probs,
            n_types=n_types,
            idx_to_type=idx_to_type,
        )
        gen_feat = magi.reconstruct_generated_features(
            gen_raw,
            # v0.8: energy comes from the mixture head as y = log10(E),
            # NOT from an energy bin index.
            energy_head_mode="mixture",
            energy_transform="log10",
            geometry_mode="quantile_u_r_u_v_phi_r_phi_v",
            qt_u_r=qt_u_r,
            qt_u_v=qt_u_v,
            qt_phi_r=qt_phi_r,
            qt_phi_v=qt_phi_v,
        )
        parts.append(magi.reconstruct_generated_physics(gen_feat, center=center, radius=R))
        done += m
        print(f"  generated {done:,}/{n_gen:,}", flush=True)

    if len(parts) == 1:
        return parts[0]
    out = {}
    for k, v in parts[0].items():
        if isinstance(v, np.ndarray) and v.ndim >= 1 and v.shape[0] == parts[0]["E_gen"].shape[0]:
            out[k] = np.concatenate([p[k] for p in parts], axis=0)
        else:
            out[k] = v
    return out


gen_phys = generate_physics(model, NGEN)

real_phys = magi.reconstruct_real_test_physics(
    # only the 4 geometry columns - energy_y and the gate slots are model-side
    X_cont_test=X_cont_test[:, :4],
    E_test_raw=E_test_raw,
    qt_u_r=qt_u_r,
    qt_u_v=qt_u_v,
    qt_phi_r=qt_phi_r,
    qt_phi_v=qt_phi_v,
    center=center,
    radius=R,
    geometry_mode="quantile_u_r_u_v_phi_r_phi_v",
)

real_phys["phi_r_real"] = np.arctan2(real_phys["sphi_r_real"], real_phys["cphi_r_real"])
real_phys["phi_v_real"] = np.arctan2(real_phys["sphi_v_real"], real_phys["cphi_v_real"])
gen_phys["phi_r_gen"] = np.arctan2(gen_phys["sphi_r_gen"], gen_phys["cphi_r_gen"])
gen_phys["phi_v_gen"] = np.arctan2(gen_phys["sphi_v_gen"], gen_phys["cphi_v_gen"])

magi.report_generated_constraints(gen_phys, radius=R)

scores = magi.compute_wasserstein_scores(real_phys, gen_phys)
print("\nWasserstein distances (real vs generated):")
for k, v in scores.items():
    print(f"  {k:8s} {v:.5f}")

In [ ]:
# ==========================================================
# Line-integral recovery: the v0.8-specific energy metric.
# recovery_ratio = (generated events near the line) / (real events near it),
# after scaling for the generated/real sample-size ratio. 1.0 = perfect.
# component_fraction = share of generated events the gate actually routed to
# that line slot, i.e. the mixture-weight the model learned.
# ==========================================================

E_real_full = feature_pack["filtered_prep"]["features"]["Energy"].to_numpy()
E_gen = gen_phys["E_gen"]

recovery = magi.compute_line_integral_recovery(
    E_real_full,
    E_gen,
    matched,
    feature_pack["energy_bins"],
    energy_component_idx_gen=gen_phys["energy_component_idx_gen"],
)

print(f"{SOURCE} line-integral recovery:")
for r in recovery:
    flag = "" if 0.8 <= r["recovery_ratio"] <= 1.25 else "   <-- off"
    print(f"  {r['label']:20s} n_real={r['n_real']:8d} n_gen={r['n_gen']:8d} "
          f"recovery={r['recovery_ratio']:.3f} "
          f"real_frac={r['real_fraction']:.5f} comp_frac={r['component_fraction']:.5f}{flag}")

pd.DataFrame(recovery)[
    ["label", "n_real", "n_gen", "recovery_ratio", "real_fraction", "component_fraction"]
]

In [ ]:
# ==========================================================
# Energy spectrum, real vs generated, on the detection binning.
# Log-log: this is the plot where the continuum shape (Compton edge, muon
# structure, low-energy tail) and the line spikes are both visible.
# ==========================================================

bins = np.asarray(feature_pack["energy_bins"])
real_counts, _ = np.histogram(E_real_full, bins=bins)
gen_counts, _ = np.histogram(E_gen, bins=bins)
scale = E_real_full.size / max(E_gen.size, 1)
centres = 0.5 * (bins[:-1] + bins[1:])

fig, ax = plt.subplots(figsize=(11, 5))
ax.step(centres, real_counts, where="mid", color="#2b6cb0", label="real")
ax.step(centres, gen_counts * scale, where="mid", color="#dd6b20", alpha=0.85,
        label=f"generated x{scale:.1f} (flow + coupling)")
for m in matched:
    ax.axvline(m["candidate_energy_mev"], color="#38a169", lw=0.7, ls="--")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Energy [MeV]")
ax.set_ylabel("counts / bin")
ax.set_title(f"{SOURCE}: real vs generated energy spectrum "
             f"(v0.8 flow + CDF warp + focal gate, {globals().get('n_epochs_run', EPOCHS)} ep)")
ax.legend(fontsize=9)
fig.savefig(f"./Plots/v0_8_real_{SOURCE}_spectrum.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
magi.set_plot_theme("light")

# ==========================================================
# Energy
# ==========================================================

magi.compare_hist_with_residuals(
    real_phys["logE_real"],
    gen_phys["logE_gen"],
    "log10(E)",
    bins=400,
    savepath=f"./Plots/v0_8_energy_distribution_{SOURCE}.png",
    dpi=300,
    show=True,
)

# ==========================================================
# Quantile-space variables, model native space
# ==========================================================

for key, label in [
    ("u_r_q", "u_r_q"),
    ("u_v_q", "u_v_q"),
    ("phi_r_q", "phi_r_q"),
    ("phi_v_q", "phi_v_q"),
]:
    magi.compare_hist_with_residuals(
        real_phys[f"{key}_real"],
        gen_phys[f"{key}_gen"],
        label,
        ratio_clip=5,
        bins=50,
    )

# ==========================================================
# Physical spherical variables
# ==========================================================

for key, label in [
    ("u_r", "u_r = cos(theta_r)"),
    ("u_v", "u_v = cos(theta_v)"),
]:
    plot_range = (
        min(real_phys[f"{key}_real"].min(), gen_phys[f"{key}_gen"].min()),
        max(real_phys[f"{key}_real"].max(), gen_phys[f"{key}_gen"].max()),
    )
    magi.compare_hist_with_residuals(
        real_phys[f"{key}_real"],
        gen_phys[f"{key}_gen"],
        label,
        range_=plot_range,
        ratio_clip=5,
        bins=50,
    )

# ==========================================================
# Physical angular variables
# ==========================================================

for key, label in [("phi_r", "phi_r"), ("phi_v", "phi_v")]:
    magi.compare_hist_with_residuals(
        real_phys[f"{key}_real"],
        gen_phys[f"{key}_gen"],
        label,
        range_=(-np.pi, np.pi),
        ratio_clip=5,
        bins=80,
    )

# ==========================================================
# Cartesian position and direction
# ==========================================================

for key in ["x", "y", "z"]:
    magi.compare_hist_with_residuals(
        real_phys[f"{key}_real"], gen_phys[f"{key}_gen"], key, ratio_clip=5, bins=50,
    )

for key in ["vx", "vy", "vz"]:
    magi.compare_hist_with_residuals(
        real_phys[f"{key}_real"], gen_phys[f"{key}_gen"], key,
        range_=(-1, 1), ratio_clip=5, bins=50,
    )

# ==========================================================
# Final reports
# ==========================================================

magi.report_final_ranges(real_phys, gen_phys)
magi.report_norm_checks(real_phys, gen_phys)

In [ ]:
# ==========================================================
# Joint structure: pairgrid over model-native + physical variables
# ==========================================================

df_real_plot, df_gen_plot, df_compare = magi.build_real_generated_featureframes(
    real_phys,
    gen_phys,
)

corr_cols_compare = [
    "logE",
    "u_r_q", "u_v_q", "phi_r_q", "phi_v_q",
    "u_r", "u_v", "phi_r", "phi_v",
]

magi.plot_pairgrid_physics(
    df_compare,
    cols=corr_cols_compare,
    class_col="sample",
    sample_size=10000,
    lower_mode="scatter",
    bins=30,
    contour_levels=8,
    figsize_scale=2.4,
)

plt.show()

In [ ]:
# ==========================================================
# Covariance / correlation triptychs: real | generated | (gen - real)
#
# This is the acceptance test for the learnable coupling prior. The number to
# watch is the energy<->geometry block: with prior="gaussian" the logE rows go
# flat at generation even though the model fits them in training (aggregated-
# posterior/prior mismatch). Target: every residual within +/-0.05.
# Same figures as tools/plot_v0_8_real_corr.py.
# ==========================================================

VARS = ["logE", "u_r", "u_v", "phi_r", "phi_v"]
LABELS = [r"$\log_{10}E$", r"$u_r$", r"$u_v$", r"$\phi_r$", r"$\phi_v$"]


def plot_matrix_triptych(df_real, df_gen, name, kind="corr", savepath=None):
    Xr = df_real[VARS].to_numpy()
    Xg = df_gen[VARS].to_numpy()

    if kind == "cov":
        Mr, Mg = np.cov(Xr, rowvar=False), np.cov(Xg, rowvar=False)
        cmap, title, vlim = "viridis", "Covariance", None
    else:
        Mr, Mg = np.corrcoef(Xr, rowvar=False), np.corrcoef(Xg, rowvar=False)
        cmap, title, vlim = "coolwarm", "Pearson correlation", (-1, 1)

    diff = Mg - Mr
    dmax = np.abs(diff).max() or 1.0

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    panels = [("real", Mr, cmap, vlim),
              ("generated", Mg, cmap, vlim),
              ("gen - real", diff, "coolwarm", (-dmax, dmax))]

    for ax, (ttl, M, cm, vl) in zip(axes, panels):
        kw = {} if vl is None else {"vmin": vl[0], "vmax": vl[1]}
        im = ax.imshow(M, cmap=cm, **kw)
        ax.set_xticks(range(len(VARS)))
        ax.set_xticklabels(LABELS, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(VARS)))
        ax.set_yticklabels(LABELS, fontsize=8)
        ax.set_title(ttl, fontsize=10)
        for a in range(len(VARS)):
            for b in range(len(VARS)):
                ax.text(b, a, f"{M[a, b]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if cm == "viridis" else "black")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle(f"{name}: {title} - real vs generated (v0.8 flow + coupling)", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    if savepath:
        fig.savefig(savepath, dpi=130, bbox_inches="tight")
    plt.show()
    return Mr, Mg


corr_r, corr_g = plot_matrix_triptych(
    df_real_plot, df_gen_plot, SOURCE, "corr",
    savepath=f"./Plots/v0_8_real_{SOURCE}_corr.png")

cov_r, cov_g = plot_matrix_triptych(
    df_real_plot, df_gen_plot, SOURCE, "cov",
    savepath=f"./Plots/v0_8_real_{SOURCE}_cov.png")

worst = np.abs(corr_g - corr_r)
i, j = np.unravel_index(np.argmax(worst), worst.shape)
print(f"largest correlation residual: {VARS[i]} <-> {VARS[j]}  "
      f"real {corr_r[i, j]:+.3f} -> gen {corr_g[i, j]:+.3f}  "
      f"(delta {corr_g[i, j] - corr_r[i, j]:+.3f})")

## Saving Model

`model.to_generation_config()` is the important part: it captures everything the
loader needs to rebuild this exact architecture (line positions, continuum mode,
flow hyperparameters **and the CDF warp knots**, prior type, focal gamma, per-line
widths and their trainability). That is why the reload cell above only has to read
the JSON back instead of retyping a config dict like v0.7.2 did.

In [ ]:
# ==========================================================
# Final model saving with metadata - v0.8
# ==========================================================

save_dir = f"trained_models/v0_8_{SOURCE}"
model_name = f"mix_{SOURCE}"

save_info = magi.save_final_trained_model(
    model=model,
    save_dir=save_dir,
    model_name=model_name,
    history=history,
    model_config=model.to_generation_config(),   # self-describing, authoritative
    preprocessing_metadata=preprocessing_metadata,
    training_metadata={
        "source": SOURCE,
        "epochs_requested": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "seed": 42,
        "geometry_mode": "quantile_u_r_u_v_phi_r_phi_v",
        "energy_transform": "log10",
        "device": "cpu",
    },
    callbacks=callbacks,
    notes=(
        "v0.8 mixture energy head: conditional RQS-flow continuum (24 bins, 3 "
        "transforms) with a CDF pre-warp, plus fixed-position Gaussian lines with "
        "widths pinned to the X-IFU 4 eV FWHM resolution. Energy head and gate "
        "conditioned on the latent z (energy_flow_condition='z_cond') with a "
        "learnable conditional coupling prior p(z|cond), so energy<->geometry "
        "coupling survives generation. Gate supervised with a focal-weighted "
        "auxiliary CE (w_gate_aux=2.0, gamma=2.0) against the ~99% continuum "
        "majority. Geometry unchanged from v0.7.2 (quantile u_r/u_v/phi_r/phi_v)."
    ),
)

import joblib

joblib.dump(
    quantile_transformers,
    f"{save_dir}/{model_name}_quantile_transformers.joblib",
)

print("saved ->", save_dir)

In [ ]:
magi.save_detector_table(
    gen_phys,
    filepath=f"./checkpoints/generated_particles/v0_8_{SOURCE}_generated.txt",
)

# Generating Input Files for Geant4

`magi.generate_detector_table_to_file` now handles the v0.8 mixture energy head:
the fitted geometry transformers travel in `geometry_metadata` and the energy
transform in `energy_metadata` (`energy_head_mode="mixture"`,
`energy_transform="log10"` — the head emits `y = log10(E)`, so there are no
energy bins). The writer chunks generation to disk, drops neutrinos, and keeps
generating until `n_events` transportable particles have been written.


In [ ]:
# Geant4-format export for the v0.8 mixture head.
# Columns: ParticleName Energy X Y Z Vx Vy Vz  (same schema as v0.7.2's export).
#
# geometry_metadata carries only JSON-safe descriptors, so it is merged with the
# fitted QuantileTransformer objects that reconstruction actually needs.

# magi.generate_detector_table_to_file(
#     model=model,
#     filepath=f"./generated/geant_input_v0_8_{SOURCE}_1M.txt",
#     n_events=1_000_000,
#     type_probs=type_probs,
#     n_types=n_types,
#     idx_to_type=idx_to_type,
#     # v0.8: no categorical energy bins - the mixture head returns y = log10(E)
#     energy_bins=None,
#     energy_metadata={
#         "energy_head_mode": "mixture",
#         "energy_transform": "log10",
#     },
#     geometry_metadata={**geometry_metadata, **quantile_transformers},
#     center=center,
#     radius=R,
#     chunk_size=100_000,
#     output_format="text",
# )


In [ ]:
# Read back and eyeball the exported file
# cols_no_event_id = ["ParticleName", "Energy", "X", "Y", "Z", "Vx", "Vy", "Vz"]
# test_df = magi.load_detector_table(
#     f"./generated/geant_input_v0_8_{SOURCE}_1M.txt",
#     columns=cols_no_event_id,
#     drop_event_id=False,
# )
# test_df.head()


## Notes on this run

**Configuration.** Flow continuum (24 bins, 3 transforms) with a CDF pre-warp,
fixed-line Gaussians pinned to 4 eV FWHM, `energy_flow_condition="z_cond"`,
`prior="coupling"`, `w_gate_aux=2.0`, `gate_focal_gamma=2.0`, 40 epochs, CPU
(~48 min CR, ~33 min Small).

**Where v0.8 stands** (details, tables and Cycle 0 -> 1 -> 2 figures in
`docs/v0.8_v072_comparison.md`):

- **Coupling — solved and validated on real data.** Every cross-correlation over
  (logE, u_r, u_v, phi_r, phi_v) is reproduced within +/-0.04 (CR) and +/-0.02-0.03
  (Small). Small's strongest physical coupling, logE<->u_v = 0.21, comes out at
  0.19-0.21 across all cycles. This is what the learnable prior was built for.
- **Continuum — much improved (Cycle 1).** The CDF warp reproduces CR's muon notch
  (~50 MeV) and restores Small's low-energy tail, which was absent below ~0.04 MeV
  with affine standardization and now tracks down to ~0.02 MeV.
- **Lines — good on Small, heterogeneous on CR (Cycle 2).** The focal gate lifted
  Small to Al 0.98 / Cu 1.06 / 511 1.29, and CR's 511 0.50 -> 0.85 and Ni 0.20 ->
  0.34, but CR Al is still under (0.13) and Cu K-beta over (1.80): a single global
  gamma raises all lines roughly uniformly and cannot match CR's very different
  per-line rarities.
- **Next lever (deferred, machinery already in place).** Per-line
  `gate_class_weights` — boost Al/Ni, damp Cu K-beta. The constructor argument,
  the loss term and the checkpoint round-trip are all implemented; only the values
  remain to be chosen.

**Versus v0.7.2.** On the raw 1-D energy marginal v0.7.2 is still ahead — a per-bin
multinomial fits a histogram almost by construction. But its lines are box-cars one
energy bin wide (~20 keV at 511 keV, ~5000x the X-IFU resolution), so it cannot
deliver a spectroscopy-grade beta. v0.8 is the architecturally correct path with its
hardest risks now retired; **v0.7.2 stays the beta fallback** until CR line recovery
closes.

## Appendix — Flow vs Gaussian continuum (synthetic demonstration)

This section is a **self-contained synthetic demo** of the v0.8 energy head's
continuum term. It reproduces CryoSphere-Small's pathology (a single sharp
continuum peak + a sparse low-energy tail + two closely-spaced lines) and
trains the **same** model twice, changing only the continuum:

- `continuum_mode="gaussian"` — the K=1 parametric-Gaussian continuum baseline.
- `continuum_mode="flow"` — the conditional rational-quadratic-spline flow
  continuum (`core/flows.py`), here with `energy_flow_condition="cond"` and the
  line width pinned to the injected resolution.

The plot below overlays the real spectrum (grey) with each model's generated
spectrum. The Gaussian continuum cannot be both sharp at the peak and near-zero
in the empty high-energy region, so it smears into a broad hump; the flow
tracks the real shape across the peak, the sparse tail, the line spike, and the
empty high-E cutoff.

**Caveat (important):** this demo uses *noise* geometry, so it isolates the
continuum-shape question and does **not** exercise the energy<->geometry
coupling. On real data the energy head must condition on the latent `z`
(`energy_flow_condition="z_cond"`, as in the training section above) to keep that
coupling — which re-introduces the aggregated-posterior/prior mismatch and needs
the **learnable prior** before generation is faithful. See
`docs/v0.8_learnable_prior_plan.md`. So: this figure demonstrates the flow's
continuum *shape capacity*, not a finished real-data result.

The full stress tests that gate each development cycle live in
`tools/synthetic_stress_test_small_singlepeak.py` (single peak, `--sparse-tail`)
and `tools/synthetic_stress_test_cr_multimodal.py` (multimodal + rare line
cluster, `--gate-focal-gamma`).

In [ ]:
# Self-contained synthetic demo (no real data needed). Curtailed to EPOCHS=30
# for the full-notebook run (it is an illustrative appendix, not the real
# result); raise it for a cleaner figure.
import numpy as np, tensorflow as tf, matplotlib.pyplot as plt
import magi

EPOCHS_DEMO = 30
rng = np.random.default_rng(42)

# --- synthetic Small-like spectrum: sharp peak + sparse tail + 2 close lines
N_TOTAL, LINE_FRAC, N_LINES_DEMO = 30_000, 0.03, 2
cont_w = np.array([0.90, 0.10]); cont_w = cont_w / cont_w.sum() * (1 - LINE_FRAC)
n_lines = int(round(N_TOTAL * LINE_FRAC)); n_cont = N_TOTAL - n_lines
n_peak = int(round(n_cont * cont_w[0] / (1 - LINE_FRAC))); n_tail = n_cont - n_peak
E_peak = 10.0 ** rng.normal(-0.60, 0.20, n_peak)
E_tail = 10.0 ** rng.normal(-2.30, 0.60, n_tail)
line_y = np.array([-2.30, -2.268]); line_mev = 10.0 ** line_y; sig_y = 0.015
per = [n_lines // N_LINES_DEMO] * (N_LINES_DEMO - 1) + [n_lines - (n_lines // N_LINES_DEMO) * (N_LINES_DEMO - 1)]
E_lines = np.concatenate([10.0 ** rng.normal(y0, sig_y, k) for y0, k in zip(line_y, per)])
E_demo = np.concatenate([E_peak, E_tail, E_lines]); rng.shuffle(E_demo)

bins_demo = magi.build_energy_bins(E_demo, mode="log_fixed_count", n_bins=256, min_counts=20)
matched_demo = [{"label": f"line_{i}", "origin": "synthetic",
                 "candidate_energy_mev": float(e), "count": int(c)}
                for i, (e, c) in enumerate(zip(line_mev, per))]
gate_t = magi.build_gate_targets(E_demo, bins_demo, matched_demo)
energy_y_demo = np.log10(E_demo).astype(np.float32); n = E_demo.size
noise = lambda: rng.normal(0, 1, n).astype(np.float32)
y_cont = np.concatenate([noise()[:, None], noise()[:, None], noise()[:, None],
                         noise()[:, None], energy_y_demo[:, None], gate_t.astype(np.float32)], 1)
cond_demo = np.ones((n, 1), np.float32)
ds = tf.data.Dataset.from_tensor_slices(
        ((y_cont, np.zeros(n, np.int32), cond_demo), np.zeros((n, 1), np.float32))
     ).shuffle(n, seed=42).batch(512)

def train_and_gen(**extra):
    m = magi.CVAE_MixEnergy_ContPhi_TaskAdaptive(
            n_types=1, line_positions_y=line_y.astype(np.float32),
            latent_dim=8, hidden=(128, 128, 64), beta=0.2, **extra)
    magi.compile_model(m, learning_rate=2e-4)
    m.fit(ds, epochs=EPOCHS_DEMO, verbose=0)
    return 10.0 ** m.generate(tf.constant(cond_demo), n)["energy_y"].numpy()

E_gauss = train_and_gen()
E_flow  = train_and_gen(continuum_mode="flow", energy_flow_condition="cond",
                        continuum_flow_y_mean=float(energy_y_demo.mean()),
                        continuum_flow_y_scale=float(energy_y_demo.std()),
                        line_logsigma_init=float(np.log(sig_y)),
                        line_logsigma_trainable=False)

# --- plot
hb = np.linspace(-3.6, 1.2, 120); rl = np.log10(E_demo)
fig, ax = plt.subplots(1, 2, figsize=(14, 5.2), sharex=True, sharey=True)
for a, Eg, ttl, c in [(ax[0], E_gauss, "Gaussian continuum (K=1 baseline)", "#d1495b"),
                      (ax[1], E_flow, "Flow continuum (new)", "#2e86ab")]:
    a.hist(rl, hb, histtype="stepfilled", color="0.75", edgecolor="0.5", label="real", log=True)
    a.hist(np.log10(Eg), hb, histtype="step", color=c, lw=2, label="generated", log=True)
    for ly in line_y: a.axvline(ly, color="0.3", ls=":", lw=1)
    a.axvspan(-0.8, -0.4, color="orange", alpha=0.10)
    a.set_title(ttl); a.set_xlabel(r"$\log_{10}(E\,/\,\mathrm{MeV})$")
    a.legend(loc="upper left", frameon=False); a.grid(alpha=0.2)
ax[0].set_ylabel("counts / bin")
fig.suptitle("v0.8 energy head on synthetic Small-like spectrum "
             "(shaded = true peak core; dotted = line positions)")
fig.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()